# Demo — A Data Contract that Carries Policy

A data contract turns governance intent into something a pipeline can **validate**. This
walkthrough shows the technique: start from an ungoverned contract, watch it fail validation,
add a **governance block**, and watch it pass. The exercise asks you to govern the Trailhead
`orders` contract and explain how a policy change propagates.

In [ ]:
from governance_toolkit import validate_contract

contract = {
    "name": "orders",
    "domain": "E-commerce Orders",
    "owner": "orders-team@trailhead.example",
    "version": "1.0.0",
    "columns": [
        {"name": "order_id", "type": "integer"},
        {"name": "customer_id", "type": "integer"},
        {"name": "order_ts", "type": "timestamp"},
        {"name": "status", "type": "string"},
        {"name": "amount", "type": "double"},
        {"name": "currency", "type": "string"},
    ],
}
ok, errs = validate_contract(contract)
print("Valid?", ok, "| errors:", errs)   # fails: no governance block yet

The validator rejects it: a contract without a governance block carries no policy. Add one (PII tags, retention, quality SLOs):

In [ ]:
for col in contract["columns"]:
    col["pii"] = col["name"] in ("customer_id",)   # FK to a PII entity
contract["governance"] = {
    "pii_tags": ["customer_id"],
    "retention": "P7Y",
    "quality_slos": [
        {"dimension": "completeness", "threshold": 0.99},
        {"dimension": "validity", "threshold": 0.98},
        {"dimension": "accuracy", "threshold": 0.995},
    ],
}
ok, errs = validate_contract(contract)
print("Valid now?", ok, "| errors:", errs)
contract["governance"]

**Takeaway / your turn:** a governed contract makes policy machine-checkable — a CI
gate can refuse to publish a product whose contract fails. In the exercise you'll govern the
`orders` contract yourself and explain how a retention change propagates to producers and
consumers.